# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Seif-2/ML-week-1-FLY/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Task shape:** "which ones first?" ranking, with an observed yes/no label underneath (`underperform_flag`, same one from w01–w04). Per the method table: start with **Logistic Regression** (readable), then **Random Forest** (stronger), evaluated at **Precision@K** since this is a ranking problem, plus ROC AUC as a general-quality check.

**A circularity problem worth naming up front, before training anything:** my w04 baseline's score is `ctr_gap * impressions_90d`, and `underperform_flag` is *defined* as `ctr_gap > 0`. That means the baseline doesn't really "predict" the label — it directly measures the same quantity (`ctr`) the label is thresholded from. It will score close to perfect by construction, not because it's a good model. That's not cheating (CTR is a legitimate, directly-observable feature per my w03 field contract), but it makes "beat the baseline" a hollow goal here.

**So the real modeling question I'm asking instead:** can I predict `underperform_flag` using ONLY features that do **not** include `ctr` or anything derived from it — i.e., the same "honest feature" discipline from w03's leakage trap? That's a genuinely harder, more useful question: *before* carefully measuring a page's CTR, can cheaper, indirect signals (position, traffic volume, content age, freshness, word count, content type, intent) already hint at which pages are likely underperforming? I still compare against the literal w04 baseline on the same split and metric — I just explain honestly why it starts from an unfair advantage.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

**Grouped by `client_id`, 70/30.** Per the data dictionary, `client_id` exists specifically for grouped train/test splits — pages from the same client likely share writing style, template, and audience quirks, so a plain random row-split could let the model "cheat" by seeing near-duplicate patterns from the same client in both train and test. `GroupShuffleSplit` guarantees zero client overlap between the two sets, confirmed below.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)].copy()

tier_median = visible.groupby("position_tier")["ctr"].transform("median")
visible["tier_median_ctr"] = tier_median
visible["ctr_gap"] = tier_median - visible["ctr"]
visible["underperform_flag"] = (visible["ctr_gap"] > 0).astype(int)
visible["baseline_score"] = visible["ctr_gap"] * visible["impressions_90d"]

print(f"Visible pool: {len(visible)} pages, base rate (underperform_flag=1): {visible['underperform_flag'].mean():.3f}")

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(visible, groups=visible["client_id"]))
train, test = visible.iloc[train_idx].copy(), visible.iloc[test_idx].copy()

overlap = set(train["client_id"]) & set(test["client_id"])
print(f"Train: {len(train)} rows, {train['client_id'].nunique()} clients")
print(f"Test:  {len(test)} rows, {test['client_id'].nunique()} clients")
print(f"Client overlap between train and test: {len(overlap)}  (0 = clean grouped split)")


Visible pool: 16726 pages, base rate (underperform_flag=1): 0.475
Train: 15265 rows, 19 clients
Test:  1461 rows, 9 clients
Client overlap between train and test: 0  (0 = clean grouped split)


## 3. Train + compare vs my baseline

Baseline evaluated on the TEST split using its literal w04 score. Logistic Regression and Random Forest trained on TRAIN using only the honest, non-`ctr` feature set (`impressions_90d`, `avg_position`, `content_age_days`, `days_since_last_update`, `word_count`, `content_type`, `main_intent`, `position_tier`), evaluated on the same TEST split, same metrics.


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K_VALUES = (50, 200)
results = []

# --- Baseline (from w04), evaluated on TEST ---
base_rate = test["underperform_flag"].mean()
baseline_auc = roc_auc_score(test["underperform_flag"], test["baseline_score"])
row = {"method": "Baseline (ctr_gap x impressions)", "roc_auc": baseline_auc}
for k in K_VALUES:
    row[f"precision@{k}"] = precision_at_k(test["baseline_score"], test["underperform_flag"].values, k)
results.append(row)

# --- Honest feature set: no ctr, no ctr_gap, no trend_* ---
num_feats = ["impressions_90d", "avg_position", "content_age_days", "days_since_last_update", "word_count"]
cat_feats = ["content_type", "main_intent", "position_tier"]

Xtr = pd.get_dummies(train[num_feats + cat_feats], columns=cat_feats).fillna(0)
Xte = pd.get_dummies(test[num_feats + cat_feats], columns=cat_feats).fillna(0)
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
ytr, yte = train["underperform_flag"], test["underperform_flag"]

scaler = StandardScaler()
Xtr_scaled = scaler.fit_transform(Xtr)
Xte_scaled = scaler.transform(Xte)

# --- Logistic Regression ---
lr = LogisticRegression(max_iter=2000, random_state=42).fit(Xtr_scaled, ytr)
lr_scores = lr.predict_proba(Xte_scaled)[:, 1]
row = {"method": "Logistic Regression (honest features)", "roc_auc": roc_auc_score(yte, lr_scores)}
for k in K_VALUES:
    row[f"precision@{k}"] = precision_at_k(lr_scores, yte.values, k)
results.append(row)

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr, ytr)
rf_scores = rf.predict_proba(Xte)[:, 1]
row = {"method": "Random Forest (honest features)", "roc_auc": roc_auc_score(yte, rf_scores)}
for k in K_VALUES:
    row[f"precision@{k}"] = precision_at_k(rf_scores, yte.values, k)
results.append(row)

results_df = pd.DataFrame(results).round(3)
print(f"Base rate on TEST split: {base_rate:.3f}\n")
print("Random seed 42 throughout (GroupShuffleSplit, LogisticRegression, RandomForestClassifier) — rerunning reproduces this table.\n")
results_df


Base rate on TEST split: 0.455

Random seed 42 throughout (GroupShuffleSplit, LogisticRegression, RandomForestClassifier) — rerunning reproduces this table.



,method,roc_auc,precision@50,precision@200
0,Baseline (ctr_gap x impressions),1.000,1.00,1.000
1,Logistic Regression (honest features),0.564,0.54,0.615
2,Random Forest (honest features),0.658,0.52,0.550


## 4. Errors and interpretation

**Why the baseline "wins":** it should score close to perfect, and that's expected, not impressive — see Section 1. The real comparison that matters is Random Forest vs Logistic Regression vs the base rate: does the model, using only indirect signals, beat blind guessing?

**What the model leans on** (Random Forest feature importances), and **three concrete wrong cases** read by hand below.


In [4]:
importances = pd.Series(rf.feature_importances_, index=Xtr.columns).sort_values(ascending=False)
print("Top RF feature importances:")
print(importances.head(8).round(3))
print()
print("avg_position, word_count, content_age_days, and impressions_90d dominate — all plausible:")
print("worse position and older/thinner content are believable proxies for a page that's likely")
print("underperforming its tier, even without looking at CTR directly.")
print()

# --- 3 concrete wrong cases ---
test_review = test.copy()
test_review["rf_score"] = rf_scores
test_review["rf_pred"] = (test_review["rf_score"] >= 0.5).astype(int)
wrong = test_review[test_review["rf_pred"] != test_review["underperform_flag"]]

print(f"Wrong predictions on TEST: {len(wrong)} of {len(test_review)} ({100*len(wrong)/len(test_review):.1f}%)\n")

cols = ["content_id", "position_tier", "avg_position", "impressions_90d", "word_count",
        "content_age_days", "underperform_flag", "rf_pred", "rf_score"]
sample_wrong = wrong.sample(min(3, len(wrong)), random_state=42)[cols]
print("Three concrete wrong cases:")
print(sample_wrong.to_string(index=False))
print()
print("These are hard cases because the honest features genuinely don't carry enough signal to")
print("separate them cleanly — e.g. a page can sit at a mediocre position with modest traffic and")
print("still land on either side of its tier's median CTR for reasons the model can't see without")
print("CTR itself (title quality, snippet wording, search intent match). That's the honest ceiling")
print("of predicting CTR-underperformance from pre-CTR signals alone.")
print()
print("Bottom line: Random Forest clears the base rate at Precision@200, meaning it's picking up real,")
print("if modest, signal from indirect features. It is not, and was never going to be, close to the")
print("baseline's near-perfect score — that comparison was never fair to begin with, and knowing why")
print("is the actual finding of this notebook.")


Top RF feature importances:
avg_position                 0.282
word_count                   0.184
content_age_days             0.176
impressions_90d              0.143
position_tier_deep           0.080
days_since_last_update       0.055
main_intent_transactional    0.016
position_tier_page_1         0.015
dtype: float64

avg_position, word_count, content_age_days, and impressions_90d dominate — all plausible:
worse position and older/thinner content are believable proxies for a page that's likely
underperforming its tier, even without looking at CTR directly.

Wrong predictions on TEST: 618 of 1461 (42.3%)

Three concrete wrong cases:
          content_id position_tier  avg_position  impressions_90d  word_count  content_age_days  underperform_flag  rf_pred  rf_score
content_4e2a02411ce7      striking          19.9             1249      3449.0               132                  1        0  0.412169
content_9c6d0b4a1ee8      page_3_5          22.7             2193         NaN           

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.